# Results

Every number here is read from `reports/`, written by the pipeline scripts (`run_pipeline.sh`).
Scores are OTTO Recall@20: `0.10 × clicks + 0.30 × carts + 0.60 × orders`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPORTS = ROOT / "reports"
COLS = ["clicks", "carts", "orders", "score"]


def load(name):
    return json.loads((REPORTS / name).read_text())


def table(results):
    return pd.DataFrame(results).T[COLS].astype(float).round(4)

## Data and split

In [ ]:
summary = load("data_summary.json")
print(f"events: {summary['events']:,}")
print(f"test cutoff: {pd.to_datetime(summary['test_cutoff'], unit='s')} | "
      f"ranker-week cutoff: {pd.to_datetime(summary['rank_cutoff'], unit='s')}")
pd.DataFrame({"test week": summary["test_week"], "ranker week": summary["rank_week"]})

## Main result (test week)

Baselines, the original two-stage system (v1, one blended list) and the final system
(v2: viewed-together candidates and a separate list per event type).

In [ ]:
baselines = table(load("baselines_test_week.json")["results"])
systems = table(load("evaluation_test_week_v1_v2.json")["results"])
main = pd.concat([baselines, systems.loc[["v1, one blended list", "v2, separate list per type"]]])
main

In [ ]:
ax = main["score"].drop("popular items only").plot.barh(figsize=(8, 3), title="Test-week score")
ax.set_xlim(main["score"].drop("popular items only").min() - 0.02, main["score"].max() + 0.01)
for p in ax.patches:
    ax.annotate(f"{p.get_width():.4f}", (p.get_width(), p.get_y() + p.get_height() / 2), va="center", xytext=(3, 0),
                textcoords="offset points")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## From v1 to v2, one change at a time

1. Separate list per event type instead of one blended list.
2. Viewed-together candidates (items other shoppers viewed within 24 hours of the shopper's items).

In [ ]:
systems

In [ ]:
ax = systems[["clicks", "carts", "orders"]].T.plot.bar(figsize=(9, 3.5), rot=0, title="Recall@20 by event type, test week")
ax.set_ylim(0.35, 0.7)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Choosing on held-out ranker-week sessions

Every choice was made on ranker-week sessions the rankers never trained on, on two random samples,
before the test week was scored once. The ranking of the four systems should be the same on both samples.

In [ ]:
held_out = pd.DataFrame({
    f"seed {seed}": table(load(f"evaluation_ranker_week_v1_v2_seed{seed}.json")["results"])["score"]
    for seed in (42, 7)
})
held_out["test week"] = systems["score"]
held_out

## What the v2 rankers rely on (share of LightGBM gain)

In [ ]:
rankers = load("rankers_v2.json")["targets"]
gain = pd.DataFrame({t: r["gain_share"] for t, r in rankers.items()}).fillna(0)
top = gain.loc[gain.max(axis=1).sort_values(ascending=False).index[:8]]

ax = top.iloc[::-1].plot.barh(figsize=(8, 4), title="Top features by gain share (v2 rankers)")
ax.set_xlabel("share of gain")
plt.tight_layout()
plt.show()
top.round(3)

## GRU session model

In [ ]:
pd.DataFrame({week: load(f"gru_{week}.json") for week in ("rank", "test")}).T[["sequences", "batch", "lr", "history"]]